# Qwen 1.5B Instruct Fine-Tuning

In [1]:
import torch
import dataset_downloader
from datasets import load_dataset, DatasetDict
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from trl import SFTConfig, SFTTrainer

In [2]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

In [3]:

def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    elif torch.cuda.is_available():
        return "cuda"
    else:
        return "cpu"

def check_cuda() -> bool:
    device = find_device()
    if device == "cuda":
        print(f"✅ CUDA is available. Found {torch.cuda.device_count()} device(s).")
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
        print("Will use quantizaiton")
        return True
    else:
        print(f"{'❌' if device == 'cpu' else '⚠️'} CUDA is NOT available. Skipping quantization, using {device}")
        return False

CUDA_AVAILABLE = check_cuda()
DEVICE = find_device()


⚠️ CUDA is NOT available. Skipping quantization, using mps


In [7]:
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/venky73/spam-mails-dataset"
local_dataset_uri = dataset_downloader.download_and_unzip_dataset(dataset_url, "datasets/spam-dataset-enron1")
column_names = ['id', 'label', 'text', 'class']

Download complete.
Extracting to /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/datasets...
Extraction complete.
Cleaned up: Removed /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/datasets/spam-dataset-enron1.zip
Found largest CSV in zip: /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/datasets/spam_ham_dataset.csv (5.25 MB)


In [8]:
dataset = load_dataset("csv", data_files=local_dataset_uri, split="train")

train_testvalid = dataset.train_test_split(test_size=0.2, seed=67)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=67)

dataset = DatasetDict({
    "train": train_testvalid["train"],
    "test": test_valid["test"],
    "validation": test_valid["train"]
})

mapping = {}
current_columns = dataset["train"].column_names

for i in range(len(column_names)):
    mapping[current_columns[i]] = column_names[i]

dataset = dataset.rename_columns(mapping)

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
def dataset_transform(sample):
    raw_response = 'valid' if sample['class'] == 0 else 'spam'
    raw_text = sample['text']

    sample['developer'] = 'Classify the message as either `valid` or `spam` do not add anything else into the response'
    sample['user']=  raw_text
    sample['final']=  raw_response
    sample['messages'] = [
        {
            'content': 'Classify the message as either `valid` or `spam` do not add anything else into the response',
            'role':'system'
        },
        {
            'content': raw_text,
            'role':'user'
        },
        {
            'content': raw_response,
            'role': 'assistant'
        }
    ]
    return sample

dataset = dataset.map(dataset_transform)
dataset = dataset.remove_columns(column_names)


Map:   0%|          | 0/4136 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

In [10]:
def balance_dataset_split(dataset_split):
    spam_samples = [sample for sample in dataset_split if sample['final'] == 'spam']
    valid_samples = [sample for sample in dataset_split if sample['final'] == 'valid']
    
    min_count = min(len(spam_samples), len(valid_samples))
    
    print(f"Original - Spam: {len(spam_samples)}, Valid: {len(valid_samples)}")
    print(f"Balanced - Using {min_count} samples from each class")
    
    balanced_samples = spam_samples[:min_count] + valid_samples[:min_count]
    
    import random
    random.seed(67)
    random.shuffle(balanced_samples)
    
    return balanced_samples

train_balanced = balance_dataset_split(dataset['train'])
test_balanced = balance_dataset_split(dataset['test'])
validation_balanced = balance_dataset_split(dataset['validation'])

from datasets import Dataset
dataset = DatasetDict({
    "train": Dataset.from_list(train_balanced),
    "test": Dataset.from_list(test_balanced),
    "validation": Dataset.from_list(validation_balanced)
})

print(f"\nFinal dataset sizes:")
print(f"Train: {len(dataset['train'])}")
print(f"Test: {len(dataset['test'])}")
print(f"Validation: {len(dataset['validation'])}")

Original - Spam: 1191, Valid: 2945
Balanced - Using 1191 samples from each class
Original - Spam: 162, Valid: 356
Balanced - Using 162 samples from each class
Original - Spam: 146, Valid: 371
Balanced - Using 146 samples from each class

Final dataset sizes:
Train: 2382
Test: 324
Validation: 292


In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

if CUDA_AVAILABLE:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        use_cache=False
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": DEVICE},
    dtype=torch.float16,
    use_cache=False
)


In [12]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_BIAS = "none"

In [14]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Qwen doesnt have a pad token, so we have to explicitly set it to the token from the SFTTrainer
tokenizer.chat_template = """{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- '\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>' }}
    {%- for tool in tools %}
        {{- '\n' }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- '\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n' }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}
        {{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {{- '<|im_start|>' + message.role }}
        {% generation %}
            {%- if message.content %}
                {{- '\n' + message.content }}
            {%- endif %}
            {%- for tool_call in message.tool_calls %}
                {%- if tool_call.function is defined %}
                    {%- set tool_call = tool_call.function %}
                {%- endif %}
                {{- '\n<tool_call>\n{"name": "' }}
                {{- tool_call.name }}
                {{- '", "arguments": ' }}
                {{- tool_call.arguments | tojson }}
                {{- '}\n</tool_call>' }}
            {%- endfor %}
            {{- '<|im_end|>\n' }}
        {% endgeneration %}
    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}"""


tokenizer.bos_token = "<|endoftext|>"
tokenizer.bos_token_id = 151643
tokenizer.pad_token = "<|endoftext|>"
tokenizer.pad_token_id = 151643
tokenizer.eos_token = "<|im_end|>"
tokenizer.eos_token_id = 151645

model.config.bos_token_id = 151643
model.config.pad_token_id = 151643
model.config.eos_token_id = 151645

model.generation_config.bos_token_id = 151643
model.generation_config.pad_token_id = 151643
model.generation_config.eos_token_id = [151643, 151645] # Allow stopping at both

In [ ]:
OUTPUT_DIR = "qwen_1.5b_finetune"
MAX_LENGTH = 256
if CUDA_AVAILABLE:
    print("✅ CUDA Detected: Using NVIDIA-optimized settings (8-bit optimizer, Liger kernel).")
    target_optim = "paged_adamw_8bit"
    target_liger_kernel = True  # Only on linux
else:
    print("🍎 CUDA Not Available (likely Mac/MPS): Using MPS-friendly settings (standard optimizer, No Liger).")
    target_optim = "adamw_torch"
    target_liger_kernel = False

# Define Training Arguments
training_args = SFTConfig(
    output_dir="./results",
    logging_steps=10000,
    disable_tqdm=True,
    log_level="error",
    report_to="none",

    # Training schedule / optimization
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=1,
    max_steps=400,
    learning_rate=5e-6,

    optim=target_optim,

    max_length=MAX_LENGTH,
    use_liger_kernel=target_liger_kernel,
    # fp16=True,
    bf16=False,
    fp16=DEVICE == "mps",

    activation_offloading=DEVICE == "cuda",
    gradient_checkpointing=True,
    assistant_only_loss=True,
)

🍎 CUDA Not Available (likely Mac/MPS): Using MPS-friendly settings (standard optimizer, No Liger).


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer
);

In [26]:
trainer_stats = trainer.train()

/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,8.803800
20,7.336200
30,4.988200
40,3.604700
50,1.481700
60,0.271400
70,0.140600
80,0.056900
90,0.048200
100,0.026800


In [40]:

def run_mail_classification(email_text: str):
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_id = tokenizer.eos_token_id

    messages = [
        {
            'content': 'Classify the message as either `valid` or `spam` do not add anything else into the response',
            'role':'system'
        },
        {
            'content': email_text,
            'role':'user'
        },
    ]

    # Update padding/end tokens
    model.generation_config.eos_token_id = [eos_id, im_end_id]
    model.generation_config.pad_token_id = tokenizer.pad_token_id


    # 3. Run Inference again (simplified)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(DEVICE)


    outputs = model.generate(
        inputs,
        max_new_tokens=1,
    )


    return tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)


In [ ]:
def run_mail_from_dataset_classification(dataset, index):
    inputs = tokenizer.apply_chat_template(
        dataset[index]['messages'][0:2], # remove the response
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(DEVICE)


    outputs = model.generate(
        inputs,
        max_new_tokens=1,
    )

    return tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)


In [35]:
test_dataset = dataset['test']
incorrect_samples = []
correct_samples_count = 0
incorrect_samples_count = 0
for i in range (0, min(len(test_dataset), 500)):
    print(f"--- Run {i} ---")
    output = run_mail_from_dataset_classification(test_dataset,i)
    actual = test_dataset[i]['messages'][2]['content']
    is_correct = output == actual
    print(f"Actual: {actual} Output: {output}")
    if is_correct:
        correct_samples_count += 1
    else:
        incorrect_samples_count += 1

    total_samples_count = correct_samples_count + incorrect_samples_count
    print(f"Samples total: {total_samples_count} samples correct: {correct_samples_count} samples incorrect: {incorrect_samples_count} accuracy: {correct_samples_count / total_samples_count}")
    if not is_correct:
        incorrect_samples.append({
            'content': test_dataset[i]['messages'][1]['content'],
            'actual': actual,
            'output': output
        })

    print(output)


--- Run 0 ---


/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Actual: valid Output: valid
Samples total: 1 samples correct: 1 samples incorrect: 0 accuracy: 1.0
valid
--- Run 1 ---
Actual: valid Output: valid
Samples total: 2 samples correct: 2 samples incorrect: 0 accuracy: 1.0
valid
--- Run 2 ---
Actual: valid Output: valid
Samples total: 3 samples correct: 3 samples incorrect: 0 accuracy: 1.0
valid
--- Run 3 ---
Actual: valid Output: valid
Samples total: 4 samples correct: 4 samples incorrect: 0 accuracy: 1.0
valid
--- Run 4 ---
Actual: spam Output: spam
Samples total: 5 samples correct: 5 samples incorrect: 0 accuracy: 1.0
spam
--- Run 5 ---
Actual: spam Output: spam
Samples total: 6 samples correct: 6 samples incorrect: 0 accuracy: 1.0
spam
--- Run 6 ---
Actual: spam Output: spam
Samples total: 7 samples correct: 7 samples incorrect: 0 accuracy: 1.0
spam
--- Run 7 ---
Actual: spam Output: spam
Samples total: 8 samples correct: 8 samples incorrect: 0 accuracy: 1.0
spam
--- Run 8 ---
Actual: spam Output: spam
Samples total: 9 samples correct: 

In [37]:
actual_valid_output_spam_count = 0
actual_spam_output_valid_count = 0
unallowed_value_outputs = 0

for sample in incorrect_samples:
    output = sample['output']
    actual = sample['actual']
    if output != 'valid' and output != 'spam':
        unallowed_value_outputs += 1
    else:
        if actual == 'valid':
            actual_valid_output_spam_count += 1
        else:
            actual_spam_output_valid_count += 1
    print("\n-- Sample fail: --")
    print(f"Guess: {output} Actual: {actual}")
    print(sample['content'][0:100])
print(f"actual_valid_output_spam_count: {actual_valid_output_spam_count} actual_spam_output_valid_count: {actual_spam_output_valid_count} unallowed_value_outputs:{unallowed_value_outputs}")


-- Sample fail: --
Guess: valid Actual: spam
Subject: re : oem photoshop , photoshop , font - size : 10 px ; text - transform : uppercase ; color

-- Sample fail: --
Guess: valid Actual: spam
Subject: zdrive 1 . 5 gb usb 2 . 0 portable storage @ $ 138 . 00
zdrive 1 . 5 gb usb 2 . 0 portable

-- Sample fail: --
Guess: valid Actual: spam
Subject: your specialist ' s appointment starts on the 8 th
never agaln
minnesota , which can clin

-- Sample fail: --
Guess: valid Actual: spam
Subject: re : final notice # 7 v 8477
hi again ,
i sent you an email last week and need to confirm

-- Sample fail: --
Guess: valid Actual: spam
Subject: urgent assistance
good day ,
i am kingsley muntu , decillion finacial services south afri

-- Sample fail: --
Guess: valid Actual: spam
Subject: notebookplus - carrying cases
notebook carrying
cases
standard
features
premium
featu

-- Sample fail: --
Guess: spam Actual: valid
Subject: union carbide - seadrift
daren
sitara # 415267 @ meter # 1332 - look like th

In [48]:
print(run_mail_classification("""Subject: E-mail details of the client.
    Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com", I just wanted to check if this information is correct.
    Best regards, John
"""))

/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


valid


In [49]:
print(run_mail_classification("""Subject: Obsługa języka polskiego.
    Cześć, wydaje mi się, ze język polski nie zostanie poprawnie sklasyfikowany.
    Pozdrawiam, Wojciech
"""))

/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/lora-fine-tuning/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


valid


In [51]:
print(run_mail_classification("""Subject: Obsługa języka polskiego.
    Kup nanjowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym uzytkownikiem!
    Pozdrawiam, Wojciech
"""))

spam


In [58]:
print(run_mail_classification("""Subject: Free iPhone.
    Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
"""))

spam
